In [1]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [4]:
loader = TextLoader('langchain_crewai_dataset.txt')
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size= 200, chunk_overlap= 20)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs,'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple'),
 Document(metad

In [5]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

In [6]:
retriever=vectorstore.as_retriever(search_type='mmr', search_kwargs= {"k":5})
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001D29EA65430>, search_type='mmr', search_kwargs={'k': 5})

In [7]:
## Here we are adding LLM and Prompt for Query Enhancement
import os
from dotenv import load_dotenv
load_dotenv


llm = init_chat_model(
    "gpt-4o-mini",
    model_provider="openai"
)


In [10]:
from langchain_core.output_parsers import StrOutputParser

In [11]:
decompostion_prompt = PromptTemplate.from_template(
    """ You are an assistant. Decompose the following complext question into 2 to 4 similar sub questions and for better document retrival.

    Question:"{question}"

    Sub-questions:
"""
)
decompostion_chain = decompostion_prompt | llm | StrOutputParser()

In [13]:
query = "How does Langchain use memory and agents compared to crew AI"
decompostion_question = decompostion_chain.invoke({"question":query})
print(decompostion_question)

1. What is the role of memory in Langchain, and how does it function within the framework?
2. How does Langchain implement agents, and what are their specific characteristics or capabilities?
3. What are the key features of Crew AI's memory system, and how does it compare to Langchain's memory?
4. In what ways does Crew AI utilize agents, and how do these implementations differ from those in Langchain?


In [14]:
qa_prompt= PromptTemplate.from_template(
    """Use the context below to answer the question

    context ={context}

    Question :{input}

    """
)
qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)


In [17]:
def full_query_decomposition_rag_pipeline(user_query):
    sub_qs_text = decompostion_chain.invoke({
        "question": user_query
    })

    sub_questions = [
        q.strip("-.123456789 ").strip()
        for q in sub_qs_text.split("\n")
        if q.strip()
    ]

    results = []

    for subq in sub_questions:
        docs = retriever.invoke(subq)

        result = qa_chain.invoke({
            "input": subq,
            "context": docs
        })

        results.append(f"Q: {subq}\nA: {result}")

    return "\n\n".join(results)

In [18]:
query = "How does Langchain Use memory and agents compared to crew ai"
final_answer = full_query_decomposition_rag_pipeline(query)
print(final_answer)

Q: What memory techniques does Langchain employ in its framework?
A: LangChain employs memory techniques such as **ConversationBufferMemory** and **ConversationSummaryMemory**. These modules enable the LLM to maintain awareness of previous conversation turns or to summarize long interactions.

Q: How does Langchain implement agents for task handling and automation?
A: LangChain implements agents for task handling and automation by utilizing Large Language Models (LLMs) to reason through decision-making processes. Specifically, agents determine which tools to call based on the task at hand, decide what input to provide to those tools, and figure out how to process the output generated. This allows for complex chains of thought to be managed effectively by abstracting several components like prompt management, retrieval, memory, and agent orchestration. As a result, developers can create end-to-end pipelines that seamlessly connect LLMs with various tools and APIs, enabling efficient aut